In [ ]:


class LazyArray:
    def __init__(self, base, operations=None):
        """
        Setup the LazyArray.
        base: The original list of numbers.
        operations: The list of functions to run later.
        """
        self.base = base
        # If no operations provided, start with empty list
        self.operations = operations if operations is not None else []

    def map(self, func):
        """
        Add a function to the to-do list.
        Returns a NEW LazyArray.
        Does NOT run the function yet.
        """
        # Copy the list so we don't change the old one (Immutability)
        new_operations = self.operations.copy()
        new_operations.append(func)

        # Return a new object with the updated list
        return LazyArray(self.base, new_operations)

    def indexOf(self, target):
        """
        Find the index.
        This is where the code actually runs.
        """
        for i in range(len(self.base)):
            # Start with the original number
            value = self.base[i]

            # Run every function in the list in order
            for operation in self.operations:

                value = operation(value)

            # Check if we found the target
            if value == target:
                return i

        # Not found
        return -1

# Test the code
arr = LazyArray([10, 20, 30, 40, 50])

# Simple test
print(arr.map(lambda x: x * 2).indexOf(40))  # 1

# Chained test
print(arr.map(lambda x: x * 2).map(lambda x: x * 3).indexOf(240))  # 3

# Independent chains test
chain1 = arr.map(lambda x: x * 2)
chain2 = arr.map(lambda x: x + 5)
print(chain1.indexOf(40))  # 1
print(chain2.indexOf(15))  # 0

# Verify branching works
doubled = arr.map(lambda x: x * 2)
c1 = doubled.map(lambda x: x + 10)
c2 = doubled.map(lambda x: x + 20)
print(c1.indexOf(50))  # (20 * 2) + 10 = 50. Index 1
print(c2.indexOf(60))  # (20 * 2) + 20 = 60. Index 1

1
3
1
0
1
1


In [ ]:
class LazyArrayChained:
    def __init__(self, base, parent=None, func=None):
        self.base = base
        self.parent = parent
        self.func = func

    def map(self, func):
        # Link the new object to the current one (self)
        return LazyArrayChained(self.base, parent=self, func=func)

    def _collect_functions(self):
        # Recursively get all functions from parents
        if self.parent is None:
            return []

        # Get parent functions, then add mine
        parent_funcs = self.parent._collect_functions()
        return parent_funcs + [self.func]

    def indexOf(self, target):
        # Get the full list of instructions
        operations = self._collect_functions()

        for i in range(len(self.base)):
            value = self.base[i]
            for op in operations:
                value = op(value)
            if value == target:
                return i
        return -1

In [ ]:
arr2 = LazyArrayChained([10, 20, 30, 40, 50])
print(arr2.map(lambda x: x * 1).map(lambda x: x * 2).indexOf(40))  # 1


1


In [ ]:
class CallCounter:
    """Helper to count how many times a function runs."""
    def __init__(self, func, name="function"):
        self.func = func
        self.name = name
        self.call_count = 0

    def __call__(self, x):
        self.call_count += 1
        return self.func(x)

    def reset(self):
        self.call_count = 0

def test_laziness():
    arr = LazyArray([1, 2, 3, 4, 5])

    # Wrap the math function with a counter
    double = CallCounter(lambda x: x * 2, "double")
    add_ten = CallCounter(lambda x: x + 10, "add_ten")

    # 1. Call map()
    lazy1 = arr.map(double)
    # Check: Count should still be 0
    assert double.call_count == 0, "map() should not call function!"

    lazy2 = lazy1.map(add_ten)
    # Check: Count should still be 0
    assert double.call_count == 0, "map() should not call function!"
    assert add_ten.call_count == 0, "map() should not call function!"

    # 2. Call indexOf() - NOW it should run
    result = lazy2.indexOf(14)  # (2 * 2) + 10 = 14. This is at index 1.

    # It processes Index 0 (No match) -> Count goes to 1
    # It processes Index 1 (Match!) -> Count goes to 2
    assert double.call_count == 2
    assert add_ten.call_count == 2
    assert result == 1

    print("✓ Laziness verified!")

test_laziness()

✓ Laziness verified!


In [ ]:
import threading

class ThreadSafeLazyArray:
    def __init__(self, base, operations=None):
        self.base = base
        self.operations = operations if operations is not None else []
        self._cached_transformed = None

        # 1. Create a "Key" (Lock) for this object
        self._lock = threading.Lock()

    def _get_transformed_array(self):
        # 2. Check the cache first (Double-checked locking pattern)
        if self._cached_transformed is not None:
            return self._cached_transformed

        # 3. Acquire the lock before calculating
        with self._lock:
            # Check again! Someone might have finished while we were waiting for the lock.
            if self._cached_transformed is None:
                print("Calculating heavy math...")
                result = []
                for value in self.base:
                    for op in self.operations:
                        value = op(value)
                    result.append(value)
                self._cached_transformed = result

            return self._cached_transformed

    def indexOf(self, target):
        return self._get_transformed_array().index(target)